# 07 — v2.5 · The product on the contracted run (run spec v3 §1a items 3, 5, 7 + §5)

On `v25_run001`: coverage per complex and per inter-complex link band (existing PAs · IPCAs incremental · the prioritizr core on
allocatable land · outside all three) with the area-expectation row; the accounting (dissolved unions beside per-link sums; branches
as links-with-n reconciled with `branches.csv`); the act check (Bow Valley rule); Layer A / slivers / Layer B tables, T1–T4, the
headline table and the GIS export. Zero solves.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

import matplotlib.pyplot as plt
import corridors_mapstyle as ms
import corridors_director as cd
import director_core as dc
import wolverine_director as wd
for _m in (ms, cd, dc, wd):
    importlib.reload(_m)

import wolverine_postprocess as wp
importlib.reload(wp)
RUN = CFG["v25"]["run_id"]
P0 = wp.load_run(config.RESULTS_DIR / CFG["results_subdir"] / RUN)

## Coverage (with the expectation row) · Accounting

In [ ]:
cov = wp.coverage(P0)
display(cov[cov.kind != "link"].round(3))
acc = wp.accounting(P0)
lk = wp.write_run_product(P0)

## The package: the act check (Bow Valley rule), Layer A / slivers / Layer B, T1–T4, the headline, GIS

In [ ]:
wd.STYLE.update(wd.V25_STYLE)
R = wp.attach(cc.load_results(config.RESULTS_DIR / CFG["results_subdir"] / RUN))
P = wd.package(R)
bow, straddles = wd.bow_valley_check(P)
display(bow)
note = ""
if straddles:                                                # the rule (run spec v3 §8): move the southern break to the Bow Valley, record it
    wd.STYLE["act_breaks_lat"] = (wd.STYLE["bow_valley_lat"],)
    P = wd.package(R); note = f"southern act break moved to the Bow Valley ({wd.STYLE['bow_valley_lat']} N): the Banff-Yoho links straddled 51 N"
    P.act_note = note; print(note)
(R.run_dir / "postprocess" / "acts.json").write_text(json.dumps(dict(break_moved=bool(straddles), act_breaks_lat=list(wd.STYLE["act_breaks_lat"]),
    note=note, links_near_banff=bow.to_dict("records")), indent=2, default=str))

In [ ]:
tA = wd.table_complexes(P); tS = wd.table_slivers(P); tB = wd.table_corridors(P)
t1 = wd.table_links(P, "examples"); t2 = wd.table_links(P, "flagged"); t3 = wd.table_links(P, "all"); t4 = wd.table_links(P, "satisfied")
hl = wd.headline(P)
wd.export_gis(P); wd.export_complexes_gis(P)
tB

## Done — `08_director_outputs.ipynb`.